# California WUI OSM 2D building view-factor graph

This is the portable statewide runner for the 2D OpenView graph. It selects **Interface** WUI across all California counties, includes buildings within one mile, and retains an additional 500-foot blocker halo. Height and terrain remain excluded.

The expensive stages are explicit opt-ins. Source preparation is cached; graph tiles are resumable. Put the CAL FIRE WUI GeoParquet and the California OSM PBF/building GeoPackage on fast local storage before starting the full run.

In [1]:
from pathlib import Path
import json
import os

import geopandas as gpd
import pandas as pd
import pyogrio
from pyproj import Transformer

from openviewcore.network import (
    BVFGraphConfig, SoCal2DSourceConfig, SoCal2DSourceResult,
    build_bvf_graph, convert_osm_pbf_buildings,
    download_geofabrik_pbf, prepare_socal_2d_source,
)

## 0. Machine and path configuration

Paths can be changed below or supplied with `OPENVIEW_WUI_PATH`, `OPENVIEW_OSM_GPKG`, and `OPENVIEW_WORKERS`. Keep `PREPARED` and `RUNS` on local NVMe storage. The historical source-helper class retains `SoCal` in its name, but an empty county tuple makes it statewide.

In [2]:
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
RAW = PROJECT / 'data' / 'external' / 'california'
PREPARED = PROJECT / 'data' / 'processed' / 'california_wui_osm_2d'
RUNS = PROJECT / 'results' / 'california_wui_osm_2d'
for path in (RAW, PREPARED, RUNS):
    path.mkdir(parents=True, exist_ok=True)

TARGET_CRS = 'EPSG:3310'  # California Albers, metres
WUI_PATH = Path(os.environ.get(
    'OPENVIEW_WUI_PATH',
    '/Users/adamswietek/Documents/PostDoc/data/raw_la/fire/20260728T155639Z_b866/part-00000.parquet',
))
OSM_OVERRIDE = os.environ.get('OPENVIEW_OSM_GPKG')
WORKERS = int(os.environ.get('OPENVIEW_WORKERS', max(1, (os.cpu_count() or 2) // 2)))

DOWNLOAD_OSM = False           # first run only; cached files are reused
REBUILD_SOURCE = False        # source preparation is not incremental
RUN_SMOKE_TEST = False
RUN_FULL_CALIFORNIA = True   # change only after the smoke test passes

TILE_M = 2_000.0
MAX_DISTANCE_M = 152.4
CALIFORNIA_OSM_URL = (
    'https://download.geofabrik.de/north-america/us/california-latest.osm.pbf'
)
print({'workers': WORKERS, 'prepared': str(PREPARED), 'runs': str(RUNS)})

{'workers': 5, 'prepared': '/Users/adamswietek/Documents/PostDoc/openview/data/processed/california_wui_osm_2d', 'runs': '/Users/adamswietek/Documents/PostDoc/openview/results/california_wui_osm_2d'}


## 1. Resolve statewide inputs

The WUI file must contain the statewide CAL FIRE geometry and `WUI_DESC`. Geofabrik supplies California as an OSM PBF, not a statewide free GeoPackage. The notebook downloads the PBF and uses `ogr2ogr` to stream only building multipolygons into a spatially indexed GeoPackage. A prebuilt GeoPackage supplied through `OPENVIEW_OSM_GPKG` skips both steps.

In [3]:
# Local imports keep this cell safe when it is rerun independently.
from openviewcore.network.socal import (
    convert_osm_pbf_buildings, download_geofabrik_pbf,
)

if not WUI_PATH.exists():
    raise FileNotFoundError(
        f'CAL FIRE WUI file not found: {WUI_PATH}. Set OPENVIEW_WUI_PATH or edit WUI_PATH.'
    )

OSM_PBF = RAW / 'california-latest.osm.pbf'
OSM_BUILDINGS_GPKG = RAW / 'california_osm_buildings.gpkg'
osm_candidates = ([] if OSM_OVERRIDE is None else [Path(OSM_OVERRIDE)]) + [
    OSM_BUILDINGS_GPKG,
    RAW / 'california.gpkg',
    *sorted(RAW.glob('california*-free.gpkg')),
    *sorted(RAW.glob('california*latest*.gpkg')),
]
OSM_GPKG = next((path for path in osm_candidates if path.is_file()), None)
if OSM_GPKG is None and not OSM_PBF.is_file() and DOWNLOAD_OSM:
    print(f'Downloading statewide California OSM PBF to {OSM_PBF}')
    download_geofabrik_pbf(OSM_PBF, url=CALIFORNIA_OSM_URL)
if OSM_GPKG is None and OSM_PBF.is_file():
    print(f'Converting California building polygons to {OSM_BUILDINGS_GPKG}')
    OSM_GPKG = convert_osm_pbf_buildings(OSM_PBF, OSM_BUILDINGS_GPKG)
if OSM_GPKG is None:
    raise FileNotFoundError(
        f'Place the California OSM PBF or building GeoPackage in {RAW}, '
        'set OPENVIEW_OSM_GPKG, or set DOWNLOAD_OSM=True.'
    )

print({'wui': str(WUI_PATH), 'osm': str(OSM_GPKG)})

{'wui': '/Users/adamswietek/Documents/PostDoc/data/raw_la/fire/20260728T155639Z_b866/part-00000.parquet', 'osm': '/Users/adamswietek/Documents/PostDoc/openview/data/external/california/california_osm_buildings.gpkg'}


## 2. Prepare statewide WUI masks and OSM footprints

`counties=()` is the important statewide change: no county filter is applied. The analysis mask is Interface WUI + 1 mile; the source mask adds the 500-foot visibility/blocker halo. The prepared FlatGeobuf supports indexed tile reads during the graph run. This preparation step currently materializes the selected statewide footprint set in memory, so it is the stage with the largest one-time RAM requirement.

In [4]:
source_cfg = SoCal2DSourceConfig(
    target_crs=TARGET_CRS,
    wui_buffer_m=1_609.344,
    blocker_halo_m=MAX_DISTANCE_M + 0.05,
    wui_simplify_m=25.0,
    counties=(),
    wui_class_column='WUI_DESC',
    wui_classes=('Interface',),
    output_prefix='california_wui',
)
expected = {
    'wui_path': PREPARED / 'california_wui.gpkg',
    'analysis_mask_path': PREPARED / 'california_wui_analysis_mask.gpkg',
    'source_mask_path': PREPARED / 'california_wui_source_mask.gpkg',
    'buildings_path': PREPARED / 'california_wui_osm_buildings.fgb',
    'manifest_path': PREPARED / 'source_manifest.json',
}
can_reuse = all(path.exists() for path in expected.values())
if can_reuse:
    old = json.loads(expected['manifest_path'].read_text())
    cfg = old.get('config', {})
    can_reuse = (
        old.get('wui_input') == str(WUI_PATH.resolve())
        and old.get('osm_geopackage') == str(OSM_GPKG.resolve())
        and cfg.get('counties') == []
        and cfg.get('wui_classes') == ['Interface']
        and cfg.get('output_prefix') == 'california_wui'
    )
if REBUILD_SOURCE or not can_reuse:
    source_result = prepare_socal_2d_source(
        OSM_GPKG, WUI_PATH, PREPARED, config=source_cfg,
    )
else:
    source_result = SoCal2DSourceResult(output_dir=PREPARED, **expected)

analysis_mask = gpd.read_file(source_result.analysis_mask_path, layer='mask')
source_manifest = json.loads(source_result.manifest_path.read_text())
print({
    'prepared_buildings': source_manifest['building_features'],
    'selected_wui_features': source_manifest['selected_wui_features'],
    'buildings_path': str(source_result.buildings_path),
})

{'prepared_buildings': 4724046, 'selected_wui_features': 7506, 'buildings_path': '/Users/adamswietek/Documents/PostDoc/openview/data/processed/california_wui_osm_2d/california_wui_osm_buildings.fgb'}


## 3. Silver Lake two-tile smoke test

This exercises the same known comparison area used in the Southern California notebook. It verifies the statewide source, selection mask, BVH backend, output schema, and resumability before committing to the statewide run.

In [5]:
SMOKE_CENTER_LON_LAT = (-118.2702, 34.0869)  # Silver Lake
cx, cy = Transformer.from_crs(4326, TARGET_CRS, always_xy=True).transform(*SMOKE_CENTER_LON_LAT)
ix, iy = int(cx // TILE_M), int(cy // TILE_M)
smoke_bounds = (ix * TILE_M, iy * TILE_M, (ix + 2) * TILE_M, (iy + 1) * TILE_M)

base_graph_cfg = dict(
    id_column='building_id', source_id_column='source_building_id',
    target_crs=TARGET_CRS, tile_size_m=TILE_M,
    max_distance_m=MAX_DISTANCE_M, max_angle_deg=80.0,
    visibility_backend='bvh', resume=True,
)
if RUN_SMOKE_TEST:
    smoke_result = build_bvf_graph(
        source_result.buildings_path, RUNS / 'silver_lake_two_tile_test',
        config=BVFGraphConfig(**base_graph_cfg, workers=1),
        bounds=smoke_bounds, max_tiles=2, selection_mask=analysis_mask,
    )
    display(pd.read_parquet(smoke_result.building_metrics_path).describe())
else:
    print('Smoke test disabled.')

Smoke test disabled.


## 4. Full California run

Set `RUN_FULL_CALIFORNIA=True` only after the smoke test passes. Each completed 2 km tile gets its own marker, so restarting this cell continues rather than discarding completed work. Begin conservatively with physical-core-scale worker counts: every worker independently reads a tile plus halo and constructs local spatial indexes. Do not run from a network-mounted output directory.

In [6]:
FULL_RUN = RUNS / 'full_california'
if RUN_FULL_CALIFORNIA:
    full_result = build_bvf_graph(
        source_result.buildings_path, FULL_RUN,
        config=BVFGraphConfig(**base_graph_cfg, workers=WORKERS),
        selection_mask=analysis_mask,
    )
    print(full_result.manifest_path)
else:
    done_dir = FULL_RUN / 'tiles_done'
    completed = len(list(done_dir.glob('*.json'))) if done_dir.exists() else 0
    print(f'Full run disabled; {completed:,} completed tile markers already exist.')

2D BVF tiles:   0%|          | 0/15776 [00:00<?, ?tile/s]

setting thread affinity failed

setting thread affinity failed
setting thread affinity failed

setting thread affinity failed
setting thread affinity failed
setting thread affinity failed


setting thread affinity failed


setting thread affinity failed
setting thread affinity failed
setting thread affinity failed
setting thread affinity failed

setting thread affinity failed
setting thread affinity failedWarning: setting thread affinity failed


setting thread affinity failed


setting thread affinity failed


setting thread affinity failed
setting thread affinity failed
setting thread affinity failed
setting thread affinity failed
setting thread affinity failed
setting thread affinity failed
setting thread affinity failed
setting thread affinity failed
setting thread affinity failed

setting thread affinity failed
setting thread affinity failed
setting thread affinity failed






setting thread affinity failed

setting thread affinity failed

setting thread affinity failed

setting 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/Users/adamswietek/Documents/PostDoc/openview/results/california_wui_osm_2d/full_california/manifest.json


## 5. Restart-safe result check

This cell does not depend on `full_result`, so it works after a kernel restart or after copying completed results. Final graph products exist only after all tiles have completed and the reduction step has run.

In [ ]:
done_dir = FULL_RUN / 'tiles_done'
status = {'completed_tile_markers': len(list(done_dir.glob('*.json'))) if done_dir.exists() else 0}
manifest_path = FULL_RUN / 'manifest.json'
metrics_path = FULL_RUN / 'building_metrics.parquet'
edges_path = FULL_RUN / 'building_edges.parquet'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    status.update({'expected_tiles': manifest['tile_count'], 'model': manifest['model']})
print(status)
if metrics_path.exists() and edges_path.exists():
    print({
        'buildings_with_metrics': len(pd.read_parquet(metrics_path, columns=['building_id'])),
        'building_edges': len(pd.read_parquet(edges_path, columns=['building_i'])),
        'metrics_path': str(metrics_path),
        'edges_path': str(edges_path),
    })
else:
    print('Final Parquet products are not present yet; resume the full run.')